In [2]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from youtube_transcript_api import YouTubeTranscriptApi
# from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings


load_dotenv()



c:\Users\Mirha\Personal\AI Engineering\gen ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

1.Indexing


In [3]:
# 1. DOCUMENT LOAD
# video_id = "Gfr50f6ZBvo" # Example video ID 
video_id = "7ARBJQn6QkM" # Example video ID  nvidia podcast

try:
    # # Get the transcript
    # ytt_api = YouTubeTranscriptApi()
    # transcript = ytt_api.fetch(video_id)

    # # Process and print the transcript
    # # full_text = ""
    # # for entry in transcript:
    # #     full_text += entry['text'] + " "

    # print(transcript)

    ytt_api = YouTubeTranscriptApi()
    fetched_transcript = ytt_api.fetch(video_id)

    # is iterable
    transcript =  " ".join(chunk.text for chunk in fetched_transcript)
    print(transcript)

except Exception as e:
    print(f"An error occurred: {e}")

At some point, you have to believe something. 
We've reinvented computing as we know it. What is the vision for what you see coming next? We 
asked ourselves, if it can do this, how far can it go? How do we get from the robots that 
we have now to the future world that you see? Cleo, everything that moves will be 
robotic someday and it will be soon. We invested tens of billions of dollars before 
it really happened. No that's very good, you did some research! But the big breakthrough 
I would say is when we... That's Jensen Huang, and whether you know it or not
his decisions are shaping your future. He's the CEO of NVIDIA, the company that skyrocketed over the past few
years to become one of the most valuable companies in the world because they led a fundamental shift 
in how computers work unleashing this current explosion of what's possible with technology. 
"NVIDIA's done it again!" We found ourselves being one of the most important technology companies in 
the world and potentiall

In [4]:
# text splitted in documents

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])
# print(chunks[100])

In [5]:
# embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# vector_store = Chroma.from_documents(
#     embedding=embedding,
#     persist_directory="YT_RAG",
#     collection_name="AI_podcast",
#     documents = chunks
# )

# embedding_model = HuggingFaceEmbeddings(
#     model_name="Qwen/Qwen3-VL-Embedding-2B",
#     multi_process=True,
#     model_kwargs={"device": "cuda"},
#     encode_kwargs={"normalize_embeddings": True},  # Set `True` for cosine similarity
# )



embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vector_store = Chroma.from_documents(
    embedding=embedding,
    persist_directory="YT_RAG",
    collection_name="AI_podcast",
    documents = chunks
)






Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3959.35it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
data = vector_store.get()
print(data)
print(data['ids'])
print(len(data['ids']))

{'ids': ['adf8e404-dbda-4ddc-b4c4-68098d2bc1ec', 'f4143ed3-2a6a-4465-bd33-450f99fdd31e', '29a9142f-3ec7-479c-8784-da829f7b0b7a', 'e28afa03-8184-4cb6-9149-1007665e3d94', 'a2041112-6978-4dc2-81f2-7811ca16ab1a', 'a53d08be-67df-4a4c-b372-951083fc86af', '906f2bef-7c7c-45c0-b909-c5b5d5f22f4e', 'dd972620-c134-43a3-9e9c-765b8a10985a', '145eb188-06cb-4b65-9edb-88bcff4e0d8f', 'b890d117-322d-4aa4-961d-32174aaed4c7', '8e90de91-78ab-4caf-8cf9-f1b4a96417bc', '3c85998c-ae9f-4302-aa46-7d0962eca974', '6b0edef6-2b40-43e4-99ba-1591b17d5329', '6289a595-ae00-4980-b13c-64898a47f2d0', '352451ae-cac3-48b3-b7ab-0d3bb85a6894', 'cd673c27-09c0-49be-8fe8-4491f58955b9', '64a065b4-bd54-455c-9b48-25e5b8c278b7', '3945c01d-dfbe-4410-be4a-35212c1c0183', 'e989c8d1-6ff5-4583-b5a9-5af4f1299167', 'd799f76e-5932-4ca8-b059-c04c92cfef9c', '76d8878b-15df-440b-8b97-53557a56464a', '1980c83a-4792-4f41-8fea-806b20159703', '9912fa9c-9086-4ab6-b758-f33f1585c479', '3f60350e-718b-44ae-9b4b-32de063290b5', '7db63f2f-5d56-4ed3-ab69-c50364

In [7]:
data = vector_store.get(
    include=["embeddings", "documents", "metadatas"]
)

print(data['embeddings'])


# these are the (vector) ,embeddings done !!

[[-7.91076198e-02  2.49181036e-02  1.98395155e-05 ...  2.02828757e-02
   2.57652458e-02  1.37937237e-02]
 [-9.00164917e-02  2.53481995e-02 -1.02321170e-02 ...  7.31676295e-02
   5.30373603e-02 -3.88631374e-02]
 [-7.94669166e-02  3.04031819e-02  4.03879257e-03 ...  5.68364784e-02
   4.12872545e-02 -3.84414680e-02]
 ...
 [-2.24671625e-02  1.70921739e-02  1.03276167e-02 ...  4.27908339e-02
   3.59051563e-02 -1.42457131e-02]
 [-2.19774917e-02  1.77542064e-02  3.45474407e-02 ... -3.18377949e-02
   8.22073445e-02 -1.72819328e-02]
 [-3.80073898e-02  6.08645519e-03  6.13315701e-02 ...  8.80090073e-02
   2.53025885e-03  8.26945622e-03]]


2.Retriver


In [8]:
retriver = vector_store.as_retriever(search_type="similarity", search_kwargs= {"k": 4})


In [9]:
result = retriver.invoke("what is gpu?")

for r in result:
    print(r.page_content,end="\n\n\n")

time or parallel processing on a GPU. "3... 2... 1..." So Nvidia unlocks all of this new power
for video games. Why gaming first? The video games requires parallel processing for processing 
3D graphics and we chose video games because, one, we loved the application, it's a simulation 
of virtual worlds and who doesn't want to go to virtual worlds and we had the good observation 
that video games has potential to be the largest market for for entertainment ever. And it turned 
out to be true. And having it being a large market is important because the technology is complicated 
and if we had a large market, our R&D budget could be large, we could create new technology. And that 
flywheel between technology and market and greater technology was really the flywheel that 
got NVIDIA to become one of the most important technology companies in the world. It was all 
because of video games. I've heard you say that GPUs were a time machine? Yeah. Could you tell me


oh my god it's revolutioni

3.Augmentation

In [10]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.7)

prompt = PromptTemplate(
    template=""" 
    You are a helpful assistant .
    Answer only from the provided context .
    if the context is insufficient then just say I don't know.

    {context}

    Question : {question}

    """,
    input_variables=["context","question"]
)

In [14]:
question = "is the topic of CUDA discussed in this video ? if yes then what was discussed?"




In [15]:
retried_docs = retriver.invoke(question)
print(retried_docs)

[Document(id='5ce0d16d-6be1-400c-b0c1-f15fd148f678', metadata={}, page_content="power. Could you explain what the vision was that led you to create CUDA? Partly researchers\xa0\ndiscovering it, partly internal inspiration and and partly solving a problem. And you know a\xa0\nlot of interesting interesting ideas come out of that soup. You know some of it is aspiration\xa0\nand inspiration, some of it is just desperation you know. And so in the case of CUDA is very\xa0\nmuch this the same way and probably the first external ideas of using our GPUs for parallel\xa0\nprocessing emerged out of some interesting work in medical imaging a couple of researchers\xa0\nat Mass General were using it to do CT reconstruction. They were using our graphics\xa0\nprocessors for that reason and it inspired us. Meanwhile the problem that we're trying to solve\xa0\ninside our company has to do with the fact that when you're trying to create these virtual worlds\xa0\nfor video games, you would like it to be 

In [16]:
context = '\n\n'.join(doc.page_content for doc in retried_docs)
context

"power. Could you explain what the vision was that led you to create CUDA? Partly researchers\xa0\ndiscovering it, partly internal inspiration and and partly solving a problem. And you know a\xa0\nlot of interesting interesting ideas come out of that soup. You know some of it is aspiration\xa0\nand inspiration, some of it is just desperation you know. And so in the case of CUDA is very\xa0\nmuch this the same way and probably the first external ideas of using our GPUs for parallel\xa0\nprocessing emerged out of some interesting work in medical imaging a couple of researchers\xa0\nat Mass General were using it to do CT reconstruction. They were using our graphics\xa0\nprocessors for that reason and it inspired us. Meanwhile the problem that we're trying to solve\xa0\ninside our company has to do with the fact that when you're trying to create these virtual worlds\xa0\nfor video games, you would like it to be beautiful but also dynamic. Water should flow like water and\n\nour futures? Wh

In [17]:
final_prompt = prompt.invoke({'question': question, 'context': context})
final_prompt

StringPromptValue(text=" \n    You are a helpful assistant .\n    Answer only from the provided context .\n    if the context is insufficient then just say I don't know.\n\n    power. Could you explain what the vision was that led you to create CUDA? Partly researchers\xa0\ndiscovering it, partly internal inspiration and and partly solving a problem. And you know a\xa0\nlot of interesting interesting ideas come out of that soup. You know some of it is aspiration\xa0\nand inspiration, some of it is just desperation you know. And so in the case of CUDA is very\xa0\nmuch this the same way and probably the first external ideas of using our GPUs for parallel\xa0\nprocessing emerged out of some interesting work in medical imaging a couple of researchers\xa0\nat Mass General were using it to do CT reconstruction. They were using our graphics\xa0\nprocessors for that reason and it inspired us. Meanwhile the problem that we're trying to solve\xa0\ninside our company has to do with the fact that

4.Generation


In [18]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of CUDA is discussed in this video.

It was created to make it easier for programmers to use GPUs for parallel processing. Previously, researchers had to "trick" GPUs into thinking their problems were graphics problems. CUDA, a platform, allows programmers to tell the GPU what to do using familiar programming languages like C, which gives more people easier access to the computing power of GPUs. The inspiration for CUDA came from researchers discovering the use of GPUs for parallel processing (specifically in medical imaging for CT reconstruction), internal inspiration within the company, and the need to solve problems, such as creating beautiful and dynamic virtual worlds for video games. The strategy behind GeForce being a vehicle for this parallel architecture was based on reasoned hope that researchers would find it useful.


chain building


making the whole multiple task automated !


In [19]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def format_doc(retried_docs):
    context = '\n\n'.join(doc.page_content for doc in retried_docs)
    return context

parser = StrOutputParser()






In [20]:
parallel_chain = RunnableParallel({
    'context': retriver | RunnableLambda(format_doc) ,
    'question':RunnablePassthrough()
})

# parallel_chain.invoke('what is deepmind?')

main_chain = parallel_chain | prompt | llm | parser

In [22]:
main_chain.invoke("what are the topics discussed in this video?")


"The topics discussed in this video include:\n\n*   Using technology to make the future better.\n*   The current moment with AI.\n*   Key insights that led to a fundamental shift in computing.\n*   What is happening right now in relation to those insights.\n*   The vision for what is coming next.\n*   The idea of having a personal R2-D2 for one's entire life.\n*   Potential challenges with AI such as bias, toxicity, and hallucination."